# 1. Dataset

In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=200,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    n_classes=2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X Shape:", X.shape)
print("y Shape:", y.shape)
print("X_train Shape:", X_train.shape)
print("X_test Shape:", X_test.shape)

X Shape: (200, 4)
y Shape: (200,)
X_train Shape: (160, 4)
X_test Shape: (40, 4)


# 2. Sklearn 

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

sk_model = RandomForestClassifier(n_estimators=10, max_depth=3, random_state=42)

sk_model.fit(X_train, y_train)

sk_pred = sk_model.predict(X_test)

print("Sklearn Prediction:")
print(sk_pred)

print("Accuracy:", accuracy_score(y_test, sk_pred))

Sklearn Prediction:
[0 0 1 0 0 0 0 1 1 0 0 1 0 0 1 0 0 0 0 0 0 1 0 0 0 1 0 1 0 0 0 0 0 1 0 1 1
 1 1 1]
Accuracy: 0.9


# 3. Scratch Code

In [6]:
class RandomForestScratch:

    def __init__(self, n_trees=10):
        self.n_trees = n_trees
        self.models = []

    def fit(self, X, y):

        for _ in range(self.n_trees):

            # STEP 1: Bootstrap sample
            idx = np.random.choice(len(X), len(X), replace=True)

            X_boot = X[idx]
            y_boot = y[idx]

            # STEP 2: Choose random feature
            feature = np.random.randint(X.shape[1])

            # STEP 3: Choose threshold
            threshold = np.median(X_boot[:, feature])

            # STEP 4: Split data
            left = y_boot[X_boot[:, feature] <= threshold]
            right = y_boot[X_boot[:, feature] > threshold]

            # STEP 5: Majority class
            left_class = np.bincount(left).argmax() if len(left) else 0
            right_class = np.bincount(right).argmax() if len(right) else 0

            self.models.append((feature, threshold, left_class, right_class))

    def predict(self, X):

        predictions = []

        for feature, threshold, left_class, right_class in self.models:

            pred = np.where(X[:, feature] <= threshold, left_class, right_class)

            predictions.append(pred)

        predictions = np.array(predictions)

        final_predictions = []

        # Majority voting for each sample
        for i in range(len(X)):

            # Get votes from all trees for this sample
            votes = predictions[:, i]

            # Select the class with most votes
            final_predictions.append(np.bincount(votes).argmax())

        return np.array(final_predictions)

In [12]:
my_model = RandomForestScratch(n_trees=10)

my_model.fit(X_train, y_train)

my_pred = my_model.predict(X_test)

print("Scratch Prediction:")
print(my_pred)

print("Accuracy:", accuracy_score(y_test, my_pred))

Scratch Prediction:
[0 0 1 0 0 0 0 1 1 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 1 1 1 1
 1 1 1]
Accuracy: 0.875


In [13]:
print("Sklearn Accuracy:", accuracy_score(y_test, sk_pred))

print("Scratch Accuracy:", accuracy_score(y_test, my_pred))

Sklearn Accuracy: 0.9
Scratch Accuracy: 0.875


# 4. Random Forest — Important Formulas

## 1. Bootstrap Sampling

For each tree, randomly sample $N$ observations **with replacement**.

$$
D_t \sim D
$$

Each tree receives a different bootstrap dataset.

---

## 2. Random Feature Selection

At each split, only a random subset of features is considered.

$$
F_t \subseteq F
$$

This makes the trees less correlated.

---

## 3. Decision Tree

Each tree produces a prediction:

$$
h_t(x)
$$

where $t$ represents the tree.

---

## 4. Majority Voting

For classification:

$$
\hat{y}
=
\operatorname{mode}
\left\{
h_1(x),
h_2(x),
\ldots,
h_T(x)
\right\}
$$

The class receiving the most votes becomes the final prediction.

---

## 5. Random Forest

$$
RF(x)
=
\operatorname{mode}
\left(
h_1(x),h_2(x),...,h_T(x)
\right)
$$

---

## 6. Why Random Forest Works

Random Forest combines:

$$
\text{Bootstrap Sampling}
+
\text{Random Feature Selection}
+
\text{Multiple Decision Trees}
+
\text{Majority Voting}
$$

The main effect is **variance reduction** and better generalization compared with a single decision tree.